# Clases y objetos · Tipos algebraicos

Un tipo algebraico describe un dato como una de varias formas posibles: un árbol es *vacío* o *un nodo con dos subárboles*; una expresión es *una constante*, *una variable*, *una suma*… Cada forma es una clase `@dataclass(frozen=True)` (inmutable, comparable por valor y con `__match_args__` para los patrones), un alias `type` las une y las funciones que las procesan tienen un `case` por forma, casi siempre recursivo. El notebook trabaja con árboles binarios y n-arios, expresiones aritméticas y booleanas y un autómata finito, y termina con tres clases con estado: un iterador sin recursión, un paginador y el patrón *memento*. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y de [Edabit](https://edabit.com/challenges/python3).

## 1. Árbol binario: vacío o nodo

Definir un árbol binario con valores en los nodos como la unión de dos formas, `Vacio` y `Nodo(valor, izq, der)`, ambas `dataclass` inmutables. Escribir con `match` las funciones `num_nodos`, `profundidad`, `espejo` (intercambia los hijos en todos los niveles), `inorden` y `dibujar` (el árbol tumbado, con la raíz a la izquierda). Comprobar que el espejo del espejo es el árbol original y que el espejo invierte el recorrido en inorden.

In [1]:
import itertools
import random
from dataclasses import FrozenInstanceError, dataclass
from functools import cache
from math import comb, isqrt


@dataclass(frozen=True)
class Vacio:
    pass


type Arbol = Vacio | Nodo


@dataclass(frozen=True)
class Nodo:
    valor: int
    izq: Arbol = Vacio()
    der: Arbol = Vacio()


def num_nodos(a: Arbol) -> int:
    match a:
        case Vacio():
            return 0
        case Nodo(_, i, d):
            return 1 + num_nodos(i) + num_nodos(d)


def profundidad(a: Arbol) -> int:
    match a:
        case Vacio():
            return 0
        case Nodo(_, i, d):
            return 1 + max(profundidad(i), profundidad(d))


def espejo(a: Arbol) -> Arbol:
    match a:
        case Vacio():
            return a
        case Nodo(v, i, d):
            return Nodo(v, espejo(d), espejo(i))


def inorden(a: Arbol) -> list:
    match a:
        case Vacio():
            return []
        case Nodo(v, i, d):
            return inorden(i) + [v] + inorden(d)


def dibujar(a: Arbol, nivel: int = 0) -> list[str]:
    match a:
        case Vacio():
            return []
        case Nodo(v, i, d):
            return dibujar(d, nivel + 1) + ["    " * nivel + str(v)] + dibujar(i, nivel + 1)


arbol = Nodo(8, Nodo(3, Nodo(1), Nodo(6, Nodo(4), Nodo(7))), Nodo(10, der=Nodo(14, Nodo(13))))
print("\n".join(dibujar(arbol)))
print(f"\nNodos: {num_nodos(arbol)}, profundidad: {profundidad(arbol)}")
print("Inorden:          ", inorden(arbol))
print("Inorden del espejo:", inorden(espejo(arbol)))
print("\nEspejo:")
print("\n".join(dibujar(espejo(arbol))))

assert espejo(espejo(arbol)) == arbol
assert inorden(espejo(arbol)) == inorden(arbol)[::-1]
assert num_nodos(arbol) == len(inorden(arbol))
assert {Nodo(1), Nodo(1), Nodo(2)} == {Nodo(1), Nodo(2)}  # iguales por valor y hashables

        14
            13
    10
8
            7
        6
            4
    3
        1

Nodos: 9, profundidad: 4
Inorden:           [1, 3, 4, 6, 7, 8, 10, 13, 14]
Inorden del espejo: [14, 13, 10, 8, 7, 6, 4, 3, 1]

Espejo:
        1
    3
            4
        6
            7
8
    10
            13
        14


## 2. Árboles con valores en las hojas

Definir árboles cuyos valores están solo en las hojas: una `Hoja(valor)` o una `Rama(izq, der)`. Escribir `num_hojas`, `num_ramas`, `borde` (los valores de las hojas de izquierda a derecha), `mapear` (aplica una función a cada hoja conservando la forma) y `misma_forma`, que decida si dos árboles tienen la misma estructura aunque sus valores sean distintos. Comprobar con árboles aleatorios que siempre hay una hoja más que ramas.

In [2]:
@dataclass(frozen=True)
class Hoja:
    valor: object


type ArbolH = Hoja | Rama


@dataclass(frozen=True)
class Rama:
    izq: ArbolH
    der: ArbolH


def num_hojas(a: ArbolH) -> int:
    match a:
        case Hoja():
            return 1
        case Rama(i, d):
            return num_hojas(i) + num_hojas(d)


def num_ramas(a: ArbolH) -> int:
    match a:
        case Hoja():
            return 0
        case Rama(i, d):
            return 1 + num_ramas(i) + num_ramas(d)


def borde(a: ArbolH) -> list:
    match a:
        case Hoja(v):
            return [v]
        case Rama(i, d):
            return borde(i) + borde(d)


def mapear(f, a: ArbolH) -> ArbolH:
    match a:
        case Hoja(v):
            return Hoja(f(v))
        case Rama(i, d):
            return Rama(mapear(f, i), mapear(f, d))


def misma_forma(a: ArbolH, b: ArbolH) -> bool:
    match a, b:
        case Hoja(), Hoja():
            return True
        case Rama(i1, d1), Rama(i2, d2):
            return misma_forma(i1, i2) and misma_forma(d1, d2)
        case _:
            return False


islas = Rama(Rama(Hoja("El Hierro"), Hoja("La Gomera")), Rama(Hoja("La Palma"), Rama(Hoja("Tenerife"), Hoja("Gran Canaria"))))
longitudes = mapear(len, islas)
otro = Rama(Hoja(1), Rama(Rama(Hoja(2), Hoja(3)), Rama(Hoja(4), Hoja(5))))
print("Borde:", borde(islas))
print("Borde de mapear(len):", borde(longitudes))
print(f"Hojas: {num_hojas(islas)}, ramas: {num_ramas(islas)}")
print("¿Misma forma que mapear(len)?", misma_forma(islas, longitudes))
print("¿Misma forma que otro árbol con 5 hojas?", misma_forma(islas, otro))


def aleatorio_h(rng: random.Random, hojas: int) -> ArbolH:
    if hojas == 1:
        return Hoja(rng.randint(0, 9))
    k = rng.randint(1, hojas - 1)
    return Rama(aleatorio_h(rng, k), aleatorio_h(rng, hojas - k))


rng = random.Random(10)
for _ in range(300):
    t = aleatorio_h(rng, rng.randint(1, 40))
    assert num_hojas(t) == num_ramas(t) + 1 == len(borde(t))
    assert misma_forma(t, mapear(str, t)) and borde(mapear(str, t)) == [str(v) for v in borde(t)]
assert not misma_forma(islas, otro) and num_hojas(islas) == num_hojas(otro)

Borde: ['El Hierro', 'La Gomera', 'La Palma', 'Tenerife', 'Gran Canaria']
Borde de mapear(len): [9, 9, 8, 8, 12]
Hojas: 5, ramas: 4
¿Misma forma que mapear(len)? True
¿Misma forma que otro árbol con 5 hojas? False


## 3. Árboles con valores en nodos y hojas: recortar, replicar y niveles

Definir árboles con valor en las hojas, `H(valor)`, y también en los nodos internos, `N(valor, izq, der)`. Escribir `prof` (una hoja tiene profundidad 0), `recortar(a, n)`, que corte el árbol a profundidad `n` convirtiendo en hojas los nodos de ese nivel, `replicar(n, x)`, que construya el árbol completo de profundidad `n` con todos los valores iguales a `x`, y `nivel(a, k)`, con los valores del nivel `k` de izquierda a derecha. Como los árboles son inmutables, `replicar` puede compartir el mismo subárbol a ambos lados.

In [3]:
@dataclass(frozen=True)
class H:
    valor: object


type ArbolHN = H | N


@dataclass(frozen=True)
class N:
    valor: object
    izq: ArbolHN
    der: ArbolHN


def prof(a: ArbolHN) -> int:
    match a:
        case H():
            return 0
        case N(_, i, d):
            return 1 + max(prof(i), prof(d))


def recortar(a: ArbolHN, n: int) -> ArbolHN:
    match a:
        case N(v, _, _) if n == 0:
            return H(v)
        case N(v, i, d):
            return N(v, recortar(i, n - 1), recortar(d, n - 1))
        case H():
            return a


def replicar(n: int, x) -> ArbolHN:
    if n == 0:
        return H(x)
    sub = replicar(n - 1, x)
    return N(x, sub, sub)


def nivel(a: ArbolHN, k: int) -> list:
    match a:
        case H(v) | N(v, _, _) if k == 0:
            return [v]
        case N(_, i, d):
            return nivel(i, k - 1) + nivel(d, k - 1)
        case H():
            return []


a = N(9, N(3, H(2), H(4)), N(7, H(5), N(6, H(1), H(8))))
for k in range(prof(a) + 1):
    print(f"nivel {k}: {nivel(a, k)}")
for n in range(4):
    print(f"recortar(a, {n}) = {recortar(a, n)}")
r = replicar(3, "x")
print("\nreplicar(3, 'x') por niveles:", [nivel(r, k) for k in range(4)])
print("Los dos hijos son el mismo objeto:", r.izq is r.der)

for n in range(6):
    assert prof(recortar(a, n)) == min(n, prof(a))
    assert all(nivel(replicar(n, 0), k) == [0] * 2**k for k in range(n + 1))
assert recortar(a, prof(a)) == a
assert sum(len(nivel(a, k)) for k in range(prof(a) + 1)) == 9

nivel 0: [9]
nivel 1: [3, 7]
nivel 2: [2, 4, 5, 6]
nivel 3: [1, 8]
recortar(a, 0) = H(valor=9)
recortar(a, 1) = N(valor=9, izq=H(valor=3), der=H(valor=7))
recortar(a, 2) = N(valor=9, izq=N(valor=3, izq=H(valor=2), der=H(valor=4)), der=N(valor=7, izq=H(valor=5), der=H(valor=6)))
recortar(a, 3) = N(valor=9, izq=N(valor=3, izq=H(valor=2), der=H(valor=4)), der=N(valor=7, izq=H(valor=5), der=N(valor=6, izq=H(valor=1), der=H(valor=8))))

replicar(3, 'x') por niveles: [['x'], ['x', 'x'], ['x', 'x', 'x', 'x'], ['x', 'x', 'x', 'x', 'x', 'x', 'x', 'x']]
Los dos hijos son el mismo objeto: True


## 4. Árboles equilibrados

Un árbol está equilibrado por número de nodos si en cada nodo los subárboles izquierdo y derecho tienen un número de nodos que difiere como mucho en 1, y equilibrado por altura (como un AVL) si lo que difiere como mucho en 1 es su profundidad. Escribir ambas comprobaciones en una sola pasada, `desde_lista(xs)`, que construya un árbol equilibrado tomando como raíz el elemento central, y `formas(n)`, que genere todos los árboles de `n` nodos. Contar para cada `n` cuántas formas hay (los números de Catalan, $\frac{1}{n+1}\binom{2n}{n}$) y cuántas están equilibradas de cada manera.

In [4]:
def _nodos_si_equilibrado(a: Arbol) -> int | None:
    match a:
        case Vacio():
            return 0
        case Nodo(_, i, d):
            ni, nd = _nodos_si_equilibrado(i), _nodos_si_equilibrado(d)
            if ni is None or nd is None or abs(ni - nd) > 1:
                return None
            return 1 + ni + nd


def _altura_si_equilibrado(a: Arbol) -> int | None:
    match a:
        case Vacio():
            return 0
        case Nodo(_, i, d):
            hi, hd = _altura_si_equilibrado(i), _altura_si_equilibrado(d)
            if hi is None or hd is None or abs(hi - hd) > 1:
                return None
            return 1 + max(hi, hd)


def equilibrado_nodos(a: Arbol) -> bool:
    return _nodos_si_equilibrado(a) is not None


def equilibrado_altura(a: Arbol) -> bool:
    return _altura_si_equilibrado(a) is not None


def desde_lista(xs) -> Arbol:
    if not xs:
        return Vacio()
    m = len(xs) // 2
    return Nodo(xs[m], desde_lista(xs[:m]), desde_lista(xs[m + 1:]))


@cache
def formas(n: int) -> tuple[Arbol, ...]:
    if n == 0:
        return (Vacio(),)
    return tuple(Nodo(0, i, d) for k in range(n) for i in formas(k) for d in formas(n - 1 - k))


def numerar(a: Arbol, contador=None) -> Arbol:
    """Pone como valores 1, 2, 3… en inorden, para poder dibujar las formas."""
    contador = contador or itertools.count(1)
    match a:
        case Vacio():
            return a
        case Nodo(_, i, d):
            izq = numerar(i, contador)
            return Nodo(next(contador), izq, numerar(d, contador))


print(f"{'n':>2} {'formas':>7} {'por nodos':>10} {'por altura':>11}")
for n in range(11):
    fs = formas(n)
    por_nodos = sum(map(equilibrado_nodos, fs))
    por_altura = sum(map(equilibrado_altura, fs))
    print(f"{n:>2} {len(fs):>7} {por_nodos:>10} {por_altura:>11}")
    assert len(fs) == comb(2 * n, n) // (n + 1)
    assert all(equilibrado_altura(t) for t in fs if equilibrado_nodos(t))

ejemplo = numerar(next(t for t in formas(7) if equilibrado_altura(t) and not equilibrado_nodos(t)))
print("\nEquilibrado por altura pero no por nodos:")
print("\n".join(dibujar(ejemplo)))

for n in range(60):
    t = desde_lista(list(range(n)))
    assert equilibrado_nodos(t) and inorden(t) == list(range(n))

 n  formas  por nodos  por altura
 0       1          1           1
 1       1          1           1
 2       2          2           2
 3       5          1           1
 4      14          4           4
 5      42          4           6
 6     132          4           4
 7     429          1          17
 8    1430          8          32
 9    4862         16          44


10   16796         32          60



Equilibrado por altura pero no por nodos:
            7
        6
    5
        4
3
        2
    1


## 5. Árboles n-arios

Un árbol n-ario (o *rosa*) tiene un valor y una tupla de hijos, que puede estar vacía. Representar la división de Canarias en provincias, islas y algunos municipios, y escribir `num_nodos_n`, `prof_n` (una hoja tiene profundidad 1), `hojas`, `ramas` (los caminos de la raíz a cada hoja) y `dibujar_n`, que lo muestre con `├──` y `└──` como el comando `tree`.

In [5]:
@dataclass(frozen=True)
class Rosa:
    valor: str
    hijos: "tuple[Rosa, ...]" = ()


def num_nodos_n(a: Rosa) -> int:
    match a:
        case Rosa(_, hijos):
            return 1 + sum(map(num_nodos_n, hijos))


def prof_n(a: Rosa) -> int:
    match a:
        case Rosa(_, ()):
            return 1
        case Rosa(_, hijos):
            return 1 + max(map(prof_n, hijos))


def hojas(a: Rosa) -> list[str]:
    match a:
        case Rosa(v, ()):
            return [v]
        case Rosa(_, hijos):
            return [h for hijo in hijos for h in hojas(hijo)]


def ramas(a: Rosa) -> list[list[str]]:
    match a:
        case Rosa(v, ()):
            return [[v]]
        case Rosa(v, hijos):
            return [[v, *r] for hijo in hijos for r in ramas(hijo)]


def dibujar_n(a: Rosa) -> list[str]:
    lineas = [a.valor]
    for k, hijo in enumerate(a.hijos):
        ultimo = k == len(a.hijos) - 1
        primera, *resto = dibujar_n(hijo)
        lineas.append(("└── " if ultimo else "├── ") + primera)
        lineas += [("    " if ultimo else "│   ") + s for s in resto]
    return lineas


def R(valor, *hijos):
    return Rosa(valor, hijos)


canarias = R("Canarias",
             R("Las Palmas",
               R("Gran Canaria", R("Telde"), R("Gáldar"), R("Mogán")),
               R("Lanzarote", R("Haría"), R("Yaiza")),
               R("Fuerteventura")),
             R("Santa Cruz de Tenerife",
               R("Tenerife", R("La Laguna"), R("Adeje")),
               R("La Palma"),
               R("La Gomera", R("Hermigua")),
               R("El Hierro")))
print("\n".join(dibujar_n(canarias)))
print(f"\nNodos: {num_nodos_n(canarias)}, profundidad: {prof_n(canarias)}, hojas: {len(hojas(canarias))}")
for r in ramas(canarias)[:4]:
    print(" / ".join(r))

assert [r[-1] for r in ramas(canarias)] == hojas(canarias)
assert max(map(len, ramas(canarias))) == prof_n(canarias)
assert num_nodos_n(canarias) == len(dibujar_n(canarias))

Canarias
├── Las Palmas
│   ├── Gran Canaria
│   │   ├── Telde
│   │   ├── Gáldar
│   │   └── Mogán
│   ├── Lanzarote
│   │   ├── Haría
│   │   └── Yaiza
│   └── Fuerteventura
└── Santa Cruz de Tenerife
    ├── Tenerife
    │   ├── La Laguna
    │   └── Adeje
    ├── La Palma
    ├── La Gomera
    │   └── Hermigua
    └── El Hierro

Nodos: 18, profundidad: 4, hojas: 11
Canarias / Las Palmas / Gran Canaria / Telde
Canarias / Las Palmas / Gran Canaria / Gáldar
Canarias / Las Palmas / Gran Canaria / Mogán
Canarias / Las Palmas / Lanzarote / Haría


## 6. Árbol de factorización

El árbol de factorización de $n$ tiene $n$ en la raíz; si $n$ es primo es una hoja, y si no, sus hijos son los árboles de $d$ y $n/d$, donde $d$ es el mayor divisor de $n$ que no pasa de $\sqrt{n}$ (así el árbol queda lo más equilibrado posible). Construirlo con los árboles `H`/`N` del ejercicio 3, dibujarlo y comprobar para todos los números del 2 al 3000 que cada nodo es el producto de sus hijos y que el borde, ordenado, es la factorización en primos calculada por división de tentativa.

In [6]:
def arbol_factorizacion(n: int) -> ArbolHN:
    d = next((d for d in range(isqrt(n), 1, -1) if n % d == 0), None)
    match d:
        case None:
            return H(n)
        case _:
            return N(n, arbol_factorizacion(d), arbol_factorizacion(n // d))


def borde_hn(a: ArbolHN) -> list:
    match a:
        case H(v):
            return [v]
        case N(_, i, d):
            return borde_hn(i) + borde_hn(d)


def bien_formado(a: ArbolHN) -> bool:
    match a:
        case H():
            return True
        case N(v, i, d):
            return v == i.valor * d.valor and bien_formado(i) and bien_formado(d)


def dibujar_hn(a: ArbolHN, nivel: int = 0) -> list[str]:
    match a:
        case H(v):
            return ["      " * nivel + str(v)]
        case N(v, i, d):
            return dibujar_hn(d, nivel + 1) + ["      " * nivel + str(v)] + dibujar_hn(i, nivel + 1)


def factores_primos(n: int) -> list[int]:
    res, p = [], 2
    while p * p <= n:
        while n % p == 0:
            res.append(p)
            n //= p
        p += 1
    return res + [n] if n > 1 else res


for n in (360, 1001, 97):
    print(f"Árbol de {n} (borde: {borde_hn(arbol_factorizacion(n))}):")
    print("\n".join(dibujar_hn(arbol_factorizacion(n))), end="\n\n")

for n in range(2, 3001):
    t = arbol_factorizacion(n)
    assert bien_formado(t) and sorted(borde_hn(t)) == factores_primos(n)
    assert isinstance(t, H) == (factores_primos(n) == [n])

Árbol de 360 (borde: [3, 2, 3, 2, 2, 5]):
            5
      20
                  2
            4
                  2
360
                  3
            6
                  2
      18
            3

Árbol de 1001 (borde: [13, 7, 11]):
            11
      77
            7
1001
      13

Árbol de 97 (borde: [97]):
97



## 7. Expresiones aritméticas: representación y evaluación

Definir las expresiones como constantes, variables, sumas, productos y potencias de exponente natural (`Const`, `Var`, `Suma`, `Prod`, `Pot`). Darles una clase base con `__add__`, `__mul__` y `__pow__` (y sus versiones por la derecha) para escribirlas con la sintaxis de Python, y un `__str__` que ponga solo los paréntesis necesarios según la precedencia. Escribir `evaluar(e, entorno)` con un diccionario de valores de las variables y `variables(e)`. Comprobar con expresiones aleatorias que la cadena, traducida a Python, da el mismo valor que `evaluar`.

In [7]:
class Expresion:
    def __add__(self, otra):
        return Suma(self, a_expr(otra))

    def __radd__(self, otra):
        return Suma(a_expr(otra), self)

    def __mul__(self, otra):
        return Prod(self, a_expr(otra))

    def __rmul__(self, otra):
        return Prod(a_expr(otra), self)

    def __pow__(self, n: int):
        return Pot(self, n)

    def __str__(self):
        return mostrar(self)


@dataclass(frozen=True)
class Const(Expresion):
    valor: int


@dataclass(frozen=True)
class Var(Expresion):
    nombre: str


type Expr = Const | Var | Suma | Prod | Pot


@dataclass(frozen=True)
class Suma(Expresion):
    izq: Expr
    der: Expr


@dataclass(frozen=True)
class Prod(Expresion):
    izq: Expr
    der: Expr


@dataclass(frozen=True)
class Pot(Expresion):
    base: Expr
    exponente: int


def a_expr(x) -> Expr:
    return x if isinstance(x, Expresion) else Const(x)


def precedencia(e: Expr) -> int:
    match e:
        case Suma():
            return 1
        case Prod():
            return 2
        case Const(v) if v < 0:
            return 2
        case Pot():
            return 3
        case _:
            return 4


def mostrar(e: Expr) -> str:
    def hijo(h, minima):
        s = mostrar(h)
        return f"({s})" if precedencia(h) < minima else s

    match e:
        case Const(v):
            return str(v)
        case Var(nombre):
            return nombre
        case Suma(a, b):
            return f"{hijo(a, 1)} + {hijo(b, 1)}"
        case Prod(a, b):
            return f"{hijo(a, 2)} * {hijo(b, 2)}"
        case Pot(a, n):
            return f"{hijo(a, 4)}^{n}"


def evaluar(e: Expr, entorno: dict[str, int]) -> int:
    match e:
        case Const(v):
            return v
        case Var(nombre) if nombre in entorno:
            return entorno[nombre]
        case Var(nombre):
            raise NameError(f"la variable {nombre!r} no tiene valor")
        case Suma(a, b):
            return evaluar(a, entorno) + evaluar(b, entorno)
        case Prod(a, b):
            return evaluar(a, entorno) * evaluar(b, entorno)
        case Pot(a, n):
            return evaluar(a, entorno) ** n


def variables(e: Expr) -> set[str]:
    match e:
        case Const():
            return set()
        case Var(nombre):
            return {nombre}
        case Suma(a, b) | Prod(a, b):
            return variables(a) | variables(b)
        case Pot(a, _):
            return variables(a)


x, y = Var("x"), Var("y")
e = 3 * x**2 + 2 * x * y + (x + 1) * (y + -2) ** 2
print(e)
print(repr(x * (y + 1)))
print("Variables:", sorted(variables(e)))
print("Valor con x=2, y=5:", evaluar(e, {"x": 2, "y": 5}))
print((-2) * Pot(Const(-2), 3) + (x**2) ** 3)
try:
    evaluar(e, {"x": 1})
except NameError as error:
    print("NameError:", error)


def aleatoria(rng: random.Random, prof: int) -> Expr:
    if prof == 0 or rng.random() < 0.25:
        return Const(rng.randint(-3, 3)) if rng.random() < 0.4 else Var(rng.choice("xy"))
    match rng.choice(["suma", "prod", "pot"]):
        case "suma":
            return Suma(aleatoria(rng, prof - 1), aleatoria(rng, prof - 1))
        case "prod":
            return Prod(aleatoria(rng, prof - 1), aleatoria(rng, prof - 1))
        case "pot":
            return Pot(aleatoria(rng, prof - 1), rng.randint(0, 3))


rng = random.Random(7)
for _ in range(500):
    expr = aleatoria(rng, 5)
    entorno = {"x": rng.randint(-4, 4), "y": rng.randint(-4, 4)}
    assert eval(str(expr).replace("^", "**"), {}, dict(entorno)) == evaluar(expr, entorno), expr

3 * x^2 + 2 * x * y + (x + 1) * (y + -2)^2
Prod(izq=Var(nombre='x'), der=Suma(izq=Var(nombre='y'), der=Const(valor=1)))
Variables: ['x', 'y']
Valor con x=2, y=5: 59
-2 * (-2)^3 + (x^2)^3
NameError: la variable 'y' no tiene valor


## 8. Sustituir y simplificar

Escribir `sustituir(e, nombre, por)`, que cambie cada aparición de una variable por otra expresión, y `simplificar(e)`, que simplifique de abajo arriba con las reglas $0 + a = a$, $1 \cdot a = a$, $0 \cdot a = 0$, $a^0 = 1$, $a^1 = a$, $(a^m)^n = a^{mn}$, operando las constantes entre sí y juntando las constantes de un producto a la izquierda ($2 \cdot (3 \cdot a) = 6 \cdot a$). Comprobar con expresiones aleatorias que la simplificación no cambia el valor, no aumenta el tamaño y no cambia si se repite.

In [8]:
def sustituir(e: Expr, nombre: str, por: Expr) -> Expr:
    match e:
        case Var(n) if n == nombre:
            return por
        case Const() | Var():
            return e
        case Suma(a, b):
            return Suma(sustituir(a, nombre, por), sustituir(b, nombre, por))
        case Prod(a, b):
            return Prod(sustituir(a, nombre, por), sustituir(b, nombre, por))
        case Pot(a, n):
            return Pot(sustituir(a, nombre, por), n)


def _suma(a: Expr, b: Expr) -> Expr:
    match a, b:
        case Const(p), Const(q):
            return Const(p + q)
        case (Const(0), c) | (c, Const(0)):
            return c
        case _:
            return Suma(a, b)


def _prod(a: Expr, b: Expr) -> Expr:
    match a, b:
        case Const(p), Const(q):
            return Const(p * q)
        case (Const(0), _) | (_, Const(0)):
            return Const(0)
        case (Const(1), c) | (c, Const(1)):
            return c
        case Const(p), Prod(Const(q), c):
            return _prod(Const(p * q), c)
        case c, Const():
            return _prod(b, c)
        case _:
            return Prod(a, b)


def _pot(a: Expr, n: int) -> Expr:
    match a, n:
        case _, 0:
            return Const(1)
        case _, 1:
            return a
        case Const(p), _:
            return Const(p**n)
        case Pot(b, m), _:
            return Pot(b, m * n)
        case _:
            return Pot(a, n)


def simplificar(e: Expr) -> Expr:
    match e:
        case Suma(a, b):
            return _suma(simplificar(a), simplificar(b))
        case Prod(a, b):
            return _prod(simplificar(a), simplificar(b))
        case Pot(a, n):
            return _pot(simplificar(a), n)
        case _:
            return e


def tamano(e: Expr) -> int:
    match e:
        case Suma(a, b) | Prod(a, b):
            return 1 + tamano(a) + tamano(b)
        case Pot(a, _):
            return 1 + tamano(a)
        case _:
            return 1


ejemplos = [0 + 1 * x, (x + 0) * (y * 1), 2 * (3 * x), (x**2) ** 3 * 1, x * 0 + y**0, (2 + 3) * x**1 * 4, Const(2) ** 3 + y]
for ej in ejemplos:
    print(f"{str(ej):<26} → {simplificar(ej)}")
f = x**2 + 3 * x
print(f"\nsustituir({f}, 'x', y + 1) = {sustituir(f, 'x', y + 1)}")

rng = random.Random(8)
for _ in range(1000):
    expr = aleatoria(rng, 5)
    s = simplificar(expr)
    assert tamano(s) <= tamano(expr) and simplificar(s) == s, expr
    for _ in range(3):
        entorno = {"x": rng.randint(-4, 4), "y": rng.randint(-4, 4)}
        assert evaluar(s, entorno) == evaluar(expr, entorno)
        assert evaluar(sustituir(expr, "x", y + 1), entorno) == evaluar(expr, entorno | {"x": entorno["y"] + 1})

0 + 1 * x                  → x
(x + 0) * y * 1            → x * y
2 * 3 * x                  → 6 * x
(x^2)^3 * 1                → x^6
x * 0 + y^0                → 1
5 * x^1 * 4                → 20 * x
2^3 + y                    → 8 + y

sustituir(x^2 + 3 * x, 'x', y + 1) = (y + 1)^2 + 3 * (y + 1)


## 9. Derivada simbólica

Escribir `derivar(e, v)` con las reglas de la suma, del producto y de la potencia junto con la regla de la cadena: $(a^n)' = n \cdot a^{n-1} \cdot a'$. Simplificar el resultado, mostrar las derivadas de algunas expresiones (también parciales respecto de `y`) y comprobarlas con `sympy` en 300 expresiones aleatorias. Como `simplificar` no agrupa términos semejantes ($x + x$ no pasa a $2x$), la comparación se hace con la diferencia expandida.

In [9]:
import sympy


def derivar(e: Expr, v: str) -> Expr:
    match e:
        case Const():
            return Const(0)
        case Var(nombre):
            return Const(1 if nombre == v else 0)
        case Suma(a, b):
            return derivar(a, v) + derivar(b, v)
        case Prod(a, b):
            return derivar(a, v) * b + a * derivar(b, v)
        case Pot(_, 0):
            return Const(0)
        case Pot(a, n):
            return n * a ** (n - 1) * derivar(a, v)


def d(e: Expr, v: str = "x") -> Expr:
    return simplificar(derivar(e, v))


for ej in [3 * x**2 + 2 * x + 1, (x**2 + 1) ** 3, x * y**2 + y**3, (x + y) * (x + -1)]:
    print(f"{str(ej):<20} d/dx = {str(d(ej)):<28} d/dy = {d(ej, 'y')}")
print("\nSin simplificar:", derivar(3 * x**2, "x"))


def a_sympy(e: Expr):
    return sympy.sympify(str(e).replace("^", "**"))


rng = random.Random(9)
for _ in range(300):
    expr = aleatoria(rng, 4)
    for v in "xy":
        esperado = sympy.diff(a_sympy(expr), sympy.Symbol(v))
        assert sympy.expand(a_sympy(d(expr, v)) - esperado) == 0, expr

3 * x^2 + 2 * x + 1  d/dx = 6 * x + 2                    d/dy = 0
(x^2 + 1)^3          d/dx = 3 * (x^2 + 1)^2 * 2 * x      d/dy = 0
x * y^2 + y^3        d/dx = y^2                          d/dy = x * 2 * y + 3 * y^2
(x + y) * (x + -1)   d/dx = x + -1 + x + y               d/dy = x + -1

Sin simplificar: 0 * x^2 + 3 * 2 * x^1 * 1


## 10. Expresiones booleanas: tabla de verdad y tautologías

Definir fórmulas proposicionales con átomos, negación, conjunción, disyunción e implicación (`Atomo`, `No`, `Y`, `O`, `Implica`), escritas con los operadores `~`, `&`, `|` y `>>`, y un `__str__` con `¬`, `∧`, `∨` y `→` y los paréntesis justos (la implicación agrupa por la derecha). Escribir `valor(f, interpretacion)`, `atomos(f)`, `tabla_verdad(f)`, `es_tautologia(f)`, `contramodelo(f)` y `equivalentes(f, g)`. Comprobar varias tautologías conocidas (*modus ponens*, ley de Peirce, De Morgan, contraposición) y encontrar un contramodelo de la falacia de afirmar el consecuente.

In [10]:
class Formula:
    def __invert__(self):
        return No(self)

    def __and__(self, otra):
        return Y(self, otra)

    def __or__(self, otra):
        return O(self, otra)

    def __rshift__(self, otra):
        return Implica(self, otra)

    def __str__(self):
        return mostrar_f(self)


@dataclass(frozen=True)
class Atomo(Formula):
    nombre: str


type Prop = Atomo | No | Y | O | Implica


@dataclass(frozen=True)
class No(Formula):
    f: Prop


@dataclass(frozen=True)
class Y(Formula):
    izq: Prop
    der: Prop


@dataclass(frozen=True)
class O(Formula):
    izq: Prop
    der: Prop


@dataclass(frozen=True)
class Implica(Formula):
    izq: Prop
    der: Prop


def prec_f(f: Prop) -> int:
    match f:
        case Implica():
            return 1
        case O():
            return 2
        case Y():
            return 3
        case No():
            return 4
        case Atomo():
            return 5


def mostrar_f(f: Prop) -> str:
    def hijo(h, minima):
        s = mostrar_f(h)
        return f"({s})" if prec_f(h) < minima else s

    match f:
        case Atomo(nombre):
            return nombre
        case No(g):
            return "¬" + hijo(g, 4)
        case Y(a, b):
            return f"{hijo(a, 3)} ∧ {hijo(b, 3)}"
        case O(a, b):
            return f"{hijo(a, 2)} ∨ {hijo(b, 2)}"
        case Implica(a, b):
            return f"{hijo(a, 2)} → {hijo(b, 1)}"


def valor(f: Prop, i: dict[str, bool]) -> bool:
    match f:
        case Atomo(nombre):
            return i[nombre]
        case No(g):
            return not valor(g, i)
        case Y(a, b):
            return valor(a, i) and valor(b, i)
        case O(a, b):
            return valor(a, i) or valor(b, i)
        case Implica(a, b):
            return not valor(a, i) or valor(b, i)


def atomos(f: Prop) -> list[str]:
    match f:
        case Atomo(nombre):
            return [nombre]
        case No(g):
            return atomos(g)
        case Y(a, b) | O(a, b) | Implica(a, b):
            return sorted(set(atomos(a)) | set(atomos(b)))


def interpretaciones(nombres: list[str]):
    for valores in itertools.product([False, True], repeat=len(nombres)):
        yield dict(zip(nombres, valores))


def tabla_verdad(f: Prop) -> None:
    nombres = atomos(f)
    print(" ".join(nombres), "|", f)
    for i in interpretaciones(nombres):
        print(" ".join("VF"[not i[n]].ljust(len(n)) for n in nombres), "|", "VF"[not valor(f, i)])


def contramodelo(f: Prop) -> dict[str, bool] | None:
    return next((i for i in interpretaciones(atomos(f)) if not valor(f, i)), None)


def es_tautologia(f: Prop) -> bool:
    return contramodelo(f) is None


def equivalentes(f: Prop, g: Prop) -> bool:
    nombres = sorted(set(atomos(f)) | set(atomos(g)))
    return all(valor(f, i) == valor(g, i) for i in interpretaciones(nombres))


p, q, r = Atomo("p"), Atomo("q"), Atomo("r")
# En Python >> tiene más prioridad que & y |: las implicaciones van entre paréntesis.
tabla_verdad(((p >> q) & p) >> q)
print()
tautologias = {
    "modus ponens": ((p >> q) & p) >> q,
    "ley de Peirce": ((p >> q) >> p) >> p,
    "silogismo": ((p >> q) & (q >> r)) >> (p >> r),
    "tercero excluido": p | ~p,
}
for nombre, f in tautologias.items():
    print(f"{nombre:<17} {str(f):<30} tautología: {es_tautologia(f)}")
    assert es_tautologia(f)

falacia = ((p >> q) & q) >> p
print(f"\n{falacia} no es tautología; contramodelo: {contramodelo(falacia)}")
assert not es_tautologia(falacia) and not valor(falacia, contramodelo(falacia))
assert equivalentes(~(p & q), ~p | ~q) and equivalentes(~(p | q), ~p & ~q)
assert equivalentes(p >> q, ~q >> ~p) and equivalentes(p >> q, ~p | q)
assert not equivalentes(p >> q, q >> p)
assert str(p >> (q >> r)) == "p → q → r" and str((p >> q) >> r) == "(p → q) → r"

p q | (p → q) ∧ p → q
F F | V
F V | V
V F | V
V V | V

modus ponens      (p → q) ∧ p → q                tautología: True
ley de Peirce     ((p → q) → p) → p              tautología: True
silogismo         (p → q) ∧ (q → r) → p → r      tautología: True
tercero excluido  p ∨ ¬p                         tautología: True

(p → q) ∧ q → p no es tautología; contramodelo: {'p': False, 'q': True}


## 11. Autómata finito determinista

Un autómata finito determinista (AFD) tiene unos estados, un alfabeto, una tabla de transiciones `(estado, símbolo) → estado`, un estado inicial y unos estados finales; acepta una cadena si, leyéndola símbolo a símbolo desde el inicial, termina en un estado final. Crear la `dataclass` inmutable `AFD`, que valide al construirse que la tabla está completa, con `recorrido(cadena)` y `acepta(cadena)`. Construir a mano el que acepta los números binarios múltiplos de 3 (el estado es el resto módulo 3 de lo leído) y generalizarlo con `multiplos(k, base)`. Comprobarlos contra `int(cadena, base) % k` con todas las cadenas cortas.

In [11]:
from collections.abc import Mapping
from types import MappingProxyType


@dataclass(frozen=True)
class AFD:
    estados: frozenset
    alfabeto: frozenset
    transiciones: Mapping
    inicial: object
    finales: frozenset

    def __post_init__(self):
        object.__setattr__(self, "transiciones", MappingProxyType(dict(self.transiciones)))
        faltan = [(q, s) for q in self.estados for s in sorted(self.alfabeto) if (q, s) not in self.transiciones]
        if faltan:
            raise ValueError(f"faltan transiciones: {faltan}")
        if self.inicial not in self.estados or not self.finales <= self.estados:
            raise ValueError("el estado inicial y los finales deben ser estados del autómata")

    def recorrido(self, cadena: str) -> list:
        estado, camino = self.inicial, [self.inicial]
        for s in cadena:
            if s not in self.alfabeto:
                raise ValueError(f"el símbolo {s!r} no está en el alfabeto")
            estado = self.transiciones[estado, s]
            camino.append(estado)
        return camino

    def acepta(self, cadena: str) -> bool:
        return self.recorrido(cadena)[-1] in self.finales


mult3 = AFD(
    estados=frozenset({"r0", "r1", "r2"}),
    alfabeto=frozenset("01"),
    transiciones={("r0", "0"): "r0", ("r0", "1"): "r1", ("r1", "0"): "r2",
                  ("r1", "1"): "r0", ("r2", "0"): "r1", ("r2", "1"): "r2"},
    inicial="r0",
    finales=frozenset({"r0"}),
)
print("estado |  0   1")
for q in sorted(mult3.estados):
    print(f"{q:>6} | {mult3.transiciones[q, '0']:>3} {mult3.transiciones[q, '1']:>3}")
for cadena in ("1001", "1011", "1111"):
    print(f"{cadena} ({int(cadena, 2):>2}): {' → '.join(mult3.recorrido(cadena))}  acepta: {mult3.acepta(cadena)}")

for cadena, prueba in [("1021", lambda: mult3.acepta("1021")),
                       ("tabla incompleta", lambda: AFD(frozenset({0}), frozenset("ab"), {(0, "a"): 0}, 0, frozenset()))]:
    try:
        prueba()
    except ValueError as error:
        print(f"{cadena}: ValueError: {error}")


def multiplos(k: int, base: int = 2) -> AFD:
    digitos = "0123456789"[:base]
    return AFD(
        estados=frozenset(range(k)),
        alfabeto=frozenset(digitos),
        transiciones={(r, c): (r * base + int(c)) % k for r in range(k) for c in digitos},
        inicial=0,
        finales=frozenset({0}),
    )


for n in range(1, 13):
    for bits in itertools.product("01", repeat=n):
        cadena = "".join(bits)
        assert mult3.acepta(cadena) == (int(cadena, 2) % 3 == 0)
        assert multiplos(3).acepta(cadena) == mult3.acepta(cadena)
for k in range(1, 10):
    for base in (2, 3, 10):
        afd = multiplos(k, base)
        for n in range(1, 5):
            for cs in itertools.product(sorted(afd.alfabeto), repeat=n):
                cadena = "".join(cs)
                assert afd.acepta(cadena) == (int(cadena, base) % k == 0)
print("\nmultiplos(7, 10) acepta 2026, 2023 y 1001:", [multiplos(7, 10).acepta(s) for s in ("2026", "2023", "1001")])

estado |  0   1
    r0 |  r0  r1
    r1 |  r2  r0
    r2 |  r1  r2
1001 ( 9): r0 → r1 → r2 → r1 → r0  acepta: True
1011 (11): r0 → r1 → r2 → r2 → r2  acepta: False
1111 (15): r0 → r1 → r0 → r1 → r0  acepta: True
1021: ValueError: el símbolo '2' no está en el alfabeto
tabla incompleta: ValueError: faltan transiciones: [(0, 'b')]

multiplos(7, 10) acepta 2026, 2023 y 1001: [False, True, True]


## 12. Iterador que aplana sin recursión

Crear la clase `Aplanador`, un iterador (`__iter__` y `__next__`) que recorra perezosamente una lista anidada a cualquier profundidad y devuelva sus elementos en orden. En lugar de recursión usa una pila explícita de iteradores: la cima es la sublista que se está recorriendo. Debe desanidar cualquier iterable salvo las cadenas, así que funciona con generadores infinitos. Compararlo con una versión recursiva en listas aleatorias y comprobar que aplana una lista anidada 10 000 veces, donde la recursiva desborda la pila.

In [12]:
from collections.abc import Iterable


class Aplanador:
    def __init__(self, anidada: Iterable):
        self._pila = [iter(anidada)]

    def __iter__(self):
        return self

    def __next__(self):
        while self._pila:
            try:
                elemento = next(self._pila[-1])
            except StopIteration:
                self._pila.pop()
                continue
            match elemento:
                case str() | bytes():
                    return elemento
                case Iterable():
                    self._pila.append(iter(elemento))
                case _:
                    return elemento
        raise StopIteration


def aplanar_rec(x) -> list:
    match x:
        case list() | tuple():
            return [y for e in x for y in aplanar_rec(e)]
        case _:
            return [x]


datos = [1, [2, (3, [4, []])], [[["cinco"]]], 6]
print(list(Aplanador(datos)))
infinita = [0, [1, (2, [3])], itertools.count(4)]
print("Con un generador infinito dentro:", list(itertools.islice(Aplanador(infinita), 10)))

it = Aplanador([1, [2]])
assert iter(it) is it and list(it) == [1, 2] and list(it) == []

profunda = actual = []
for _ in range(10_000):
    actual.append([])
    actual = actual[0]
actual.append(42)
try:
    aplanar_rec(profunda)
except RecursionError as error:
    print(f"Recursiva: {type(error).__name__}: {error}")
print("Aplanador:", list(Aplanador(profunda)))


def anidada(rng: random.Random, prof: int) -> list:
    return [anidada(rng, prof - 1) if prof and rng.random() < 0.3 else rng.randint(0, 99)
            for _ in range(rng.randint(0, 4))]


rng = random.Random(12)
for _ in range(500):
    lista = anidada(rng, 6)
    assert list(Aplanador(lista)) == aplanar_rec(lista)

[1, 2, 3, 4, 'cinco', 6]
Con un generador infinito dentro: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Recursiva: RecursionError: maximum recursion depth exceeded
Aplanador: [42]


## 13. Paginador

Crear una clase `Paginador` que reparta una secuencia en páginas de un tamaño fijo, con la página `actual` (empezando en 1), `total_paginas`, los `elementos` de la página actual, `hay_siguiente` y `hay_anterior`, y los métodos `siguiente`, `anterior` (que no pasan de los extremos) e `ir_a(n)`, que valide el tipo y el rango. Manejarlo con una lista de órdenes de texto (`siguiente`, `anterior`, `primera`, `última`, `ir 3`) interpretadas con `match`, mostrando el error de las órdenes no válidas.

In [13]:
class Paginador:
    def __init__(self, elementos, por_pagina: int):
        if isinstance(por_pagina, bool) or not isinstance(por_pagina, int) or por_pagina < 1:
            raise ValueError("por_pagina debe ser un entero positivo")
        self._elementos = tuple(elementos)
        self._por_pagina = por_pagina
        self._actual = 1

    @property
    def total_paginas(self) -> int:
        return max(1, -(-len(self._elementos) // self._por_pagina))

    @property
    def actual(self) -> int:
        return self._actual

    @property
    def elementos(self) -> tuple:
        inicio = (self._actual - 1) * self._por_pagina
        return self._elementos[inicio:inicio + self._por_pagina]

    @property
    def hay_siguiente(self) -> bool:
        return self._actual < self.total_paginas

    @property
    def hay_anterior(self) -> bool:
        return self._actual > 1

    def ir_a(self, n: int) -> "Paginador":
        if isinstance(n, bool) or not isinstance(n, int):
            raise TypeError(f"el número de página debe ser un entero, no {type(n).__name__}")
        if not 1 <= n <= self.total_paginas:
            raise ValueError(f"la página {n} no existe (hay {self.total_paginas})")
        self._actual = n
        return self

    def siguiente(self) -> "Paginador":
        return self.ir_a(min(self._actual + 1, self.total_paginas))

    def anterior(self) -> "Paginador":
        return self.ir_a(max(self._actual - 1, 1))

    def __str__(self):
        return f"Página {self._actual}/{self.total_paginas}: {', '.join(map(str, self.elementos))}"


def ejecutar(p: Paginador, orden: str) -> None:
    match orden.split():
        case ["siguiente"]:
            p.siguiente()
        case ["anterior"]:
            p.anterior()
        case ["primera"]:
            p.ir_a(1)
        case ["última"]:
            p.ir_a(p.total_paginas)
        case ["ir", n] if n.lstrip("-").isdigit():
            p.ir_a(int(n))
        case _:
            raise ValueError(f"orden no reconocida: {orden!r}")


municipios = ["Agaete", "Agüimes", "Artenara", "Arucas", "Firgas", "Gáldar", "Ingenio", "La Aldea de San Nicolás",
              "Las Palmas de Gran Canaria", "Mogán", "Moya", "San Bartolomé de Tirajana", "Santa Brígida",
              "Santa Lucía de Tirajana", "Santa María de Guía", "Tejeda", "Telde", "Teror", "Valleseco",
              "Valsequillo", "Vega de San Mateo"]
p = Paginador(municipios, 5)
print(p)
for orden in ["siguiente", "siguiente", "última", "siguiente", "ir 2", "anterior", "anterior", "ir 9", "ir -1", "saltar"]:
    try:
        ejecutar(p, orden)
        print(f"{orden:<10} → {p}")
    except ValueError as error:
        print(f"{orden:<10} → ValueError: {error}")
for prueba in (lambda: p.ir_a("2"), lambda: Paginador(municipios, 0)):
    try:
        prueba()
    except (TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

p.ir_a(1)
paginas = [p.elementos]
while p.hay_siguiente:
    paginas.append(p.siguiente().elementos)
assert [m for pagina in paginas for m in pagina] == municipios and len(paginas) == p.total_paginas == 5
assert not p.hay_siguiente and p.siguiente().actual == 5 and p.ir_a(1).anterior().actual == 1
vacio = Paginador([], 3)
assert vacio.total_paginas == 1 and vacio.elementos == () and not vacio.hay_siguiente
for n in range(0, 30):
    for k in range(1, 8):
        assert Paginador(range(n), k).total_paginas == max(1, len(range(0, n, k)))

Página 1/5: Agaete, Agüimes, Artenara, Arucas, Firgas
siguiente  → Página 2/5: Gáldar, Ingenio, La Aldea de San Nicolás, Las Palmas de Gran Canaria, Mogán
siguiente  → Página 3/5: Moya, San Bartolomé de Tirajana, Santa Brígida, Santa Lucía de Tirajana, Santa María de Guía
última     → Página 5/5: Vega de San Mateo
siguiente  → Página 5/5: Vega de San Mateo
ir 2       → Página 2/5: Gáldar, Ingenio, La Aldea de San Nicolás, Las Palmas de Gran Canaria, Mogán
anterior   → Página 1/5: Agaete, Agüimes, Artenara, Arucas, Firgas
anterior   → Página 1/5: Agaete, Agüimes, Artenara, Arucas, Firgas
ir 9       → ValueError: la página 9 no existe (hay 5)
ir -1      → ValueError: la página -1 no existe (hay 5)
saltar     → ValueError: orden no reconocida: 'saltar'
TypeError: el número de página debe ser un entero, no str
ValueError: por_pagina debe ser un entero positivo


## 14. Patrón *memento*: deshacer y rehacer

El patrón *memento* guarda instantáneas del estado de un objeto sin exponer sus detalles: el objeto (*originador*) crea la instantánea y sabe restaurarla, y otro objeto (*cuidador*) solo las almacena. Crear un `Editor` de texto con cursor (`escribir`, `borrar` hacia atrás y `mover`), una `Instantanea` inmutable con `guardar` y `restaurar`, y un `Historial` que reciba las órdenes como tuplas (`("escribir", "Hola")`), las interprete con `match`, guarde una instantánea antes de cada una y permita `deshacer` y `rehacer`. Comprobar con 300 órdenes aleatorias que deshacerlas todas recorre los estados en orden inverso.

In [14]:
@dataclass(frozen=True)
class Instantanea:
    texto: str
    cursor: int


class Editor:
    def __init__(self):
        self._texto, self._cursor = "", 0

    def escribir(self, s: str) -> None:
        self._texto = self._texto[:self._cursor] + s + self._texto[self._cursor:]
        self._cursor += len(s)

    def borrar(self, n: int = 1) -> None:
        n = min(n, self._cursor)
        self._texto = self._texto[:self._cursor - n] + self._texto[self._cursor:]
        self._cursor -= n

    def mover(self, posicion: int) -> None:
        if not 0 <= posicion <= len(self._texto):
            raise ValueError(f"posición {posicion} fuera del texto")
        self._cursor = posicion

    def guardar(self) -> Instantanea:
        return Instantanea(self._texto, self._cursor)

    def restaurar(self, m: Instantanea) -> None:
        self._texto, self._cursor = m.texto, m.cursor

    def __str__(self):
        return self._texto[:self._cursor] + "|" + self._texto[self._cursor:]


class Historial:
    def __init__(self, editor: Editor):
        self._editor = editor
        self._deshacer: list[Instantanea] = []
        self._rehacer: list[Instantanea] = []

    def hacer(self, orden: tuple) -> None:
        antes = self._editor.guardar()
        match orden:
            case ("escribir", str(texto)):
                self._editor.escribir(texto)
            case ("borrar", int(n)) if n > 0:
                self._editor.borrar(n)
            case ("mover", int(posicion)):
                self._editor.mover(posicion)
            case _:
                raise ValueError(f"orden no válida: {orden!r}")
        self._deshacer.append(antes)
        self._rehacer.clear()

    def deshacer(self) -> bool:
        if not self._deshacer:
            return False
        self._rehacer.append(self._editor.guardar())
        self._editor.restaurar(self._deshacer.pop())
        return True

    def rehacer(self) -> bool:
        if not self._rehacer:
            return False
        self._deshacer.append(self._editor.guardar())
        self._editor.restaurar(self._rehacer.pop())
        return True


editor = Editor()
historial = Historial(editor)
for orden in [("escribir", "Hola Tenerife"), ("borrar", 8), ("escribir", "Gran Canaria"), ("mover", 5),
              ("escribir", "desde "), ("mover", 99), ("saltar", 1)]:
    try:
        historial.hacer(orden)
        print(f"{str(orden):<28} {editor}")
    except ValueError as error:
        print(f"{str(orden):<28} ValueError: {error}")
for accion in ["deshacer", "deshacer", "rehacer", "deshacer", "deshacer", "deshacer", "deshacer"]:
    hecho = getattr(historial, accion)()
    print(f"{accion:<28} {editor}{'' if hecho else '   (no hay nada que ' + accion + ')'}")

m = editor.guardar()
try:
    m.texto = "otro"
except FrozenInstanceError as error:
    print("FrozenInstanceError:", error)

rng = random.Random(14)
editor, estados = Editor(), []
historial = Historial(editor)
for _ in range(300):
    estados.append(str(editor))
    match rng.choice(["escribir", "borrar", "mover"]):
        case "escribir":
            historial.hacer(("escribir", rng.choice(["a", "be", "ce", " "])))
        case "borrar":
            historial.hacer(("borrar", rng.randint(1, 3)))
        case "mover":
            historial.hacer(("mover", rng.randint(0, len(editor.guardar().texto))))
final = str(editor)
for esperado in reversed(estados):
    assert historial.deshacer() and str(editor) == esperado
assert not historial.deshacer()
while historial.rehacer():
    pass
assert str(editor) == final

('escribir', 'Hola Tenerife') Hola Tenerife|
('borrar', 8)                Hola |
('escribir', 'Gran Canaria') Hola Gran Canaria|
('mover', 5)                 Hola |Gran Canaria
('escribir', 'desde ')       Hola desde |Gran Canaria
('mover', 99)                ValueError: posición 99 fuera del texto
('saltar', 1)                ValueError: orden no válida: ('saltar', 1)
deshacer                     Hola |Gran Canaria
deshacer                     Hola Gran Canaria|
rehacer                      Hola |Gran Canaria
deshacer                     Hola Gran Canaria|
deshacer                     Hola |
deshacer                     Hola Tenerife|
deshacer                     |
FrozenInstanceError: cannot assign to field 'texto'
